# Message Cookbook

Notebook này minh họa cách thiết lập và tận dụng khả năng sử dụng lớp `BaseMessage` của CAMEL.

Trong notebook này, bạn sẽ khám phá:
- **CAMEL**: Một khung làm việc đa tác nhân mạnh mẽ hỗ trợ Retrieval-Augmented Generation và các kịch bản đóng vai đa tác nhân, cho phép thực hiện các nhiệm vụ phức tạp do AI điều khiển.
- **BaseMessage**: Lớp cơ sở cho các đối tượng tin nhắn được sử dụng trong hệ thống trò chuyện CAMEL. Lớp này được thiết kế để cung cấp cấu trúc nhất quán cho các tin nhắn trong hệ thống và cho phép chuyển đổi dễ dàng giữa các loại tin nhắn khác nhau.

Trong hướng dẫn này, chúng ta sẽ khám phá lớp `BaseMessage`. Các chủ đề được đề cập bao gồm:

1. Giới thiệu về lớp `BaseMessage`.
2. Tạo một phiên bản `BaseMessage`.
3. Hiểu các thuộc tính của lớp `BaseMessage`.
4. Sử dụng các phương thức của lớp `BaseMessage`.
5. Gửi tin nhắn đến `ChatAgent`

In [4]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

True

### Give message to `ChatAgent` directly

In [8]:
from camel.agents import ChatAgent

# Define system message
sys_msg = "You are a helpful assistant."

# Set agent
camel_agent = ChatAgent(system_message=sys_msg, output_language="Vietnamese")

# Setup user message
user_msg = "What is the capital of France?"

response = camel_agent.step(user_msg)
print(response.msgs[0].content)


Thủ đô của Pháp là Paris.


### Give message to `ChatAgent` via `BaseMessage`

Để tạo một phiên bản `BaseMessage` , bạn cần cung cấp các đối số sau:
- `role_name` : Tên của vai trò người dùng hoặc trợ lý.
- `role_type` : Loại vai trò, có thể là `RoleType.ASSISTANT` hoặc `RoleType.USER` .
- `content` : Nội dung của tin nhắn.
- `meta_dict` : Một từ điển siêu dữ liệu cho tin nhắn.

Dưới đây là các đối số tùy chọn bạn có thể truyền vào:
- `video_bytes` : Các byte tùy chọn của một video liên quan đến tin nhắn.
- `image_list` : Danh sách tùy chọn các đối tượng PIL Image liên kết với tin nhắn.
- `image_detail` : Mức độ chi tiết của các hình ảnh đi kèm với tin nhắn. Mặc định là "`auto`".
- `video_detail` : Mức độ chi tiết của các video đi kèm với tin nhắn. Mặc định là "`low`".

Dưới đây là một ví dụ về việc tạo một phiên bản `BaseMessage` :


In [10]:
from camel.messages import BaseMessage
from camel.types import RoleType

message = BaseMessage(
    role_name="test_user",
    role_type=RoleType.USER,
    content="test content",
    meta_dict={}
)

Ngoài ra, lớp BaseMessage cung cấp các phương thức lớp để dễ dàng tạo tin nhắn cho `user` và `assistant` agent:

Tạo tin nhắn cho `user agent`:

In [11]:
from camel.messages import BaseMessage

user_message = BaseMessage.make_user_message(
    role_name="user_name",
    content="test content for user",
)

Creating an `assistant agent` message:


In [12]:
from camel.messages import BaseMessage

assistant_message = BaseMessage.make_assistant_message(
    role_name="assistant_name",
    content="test content for assistant",
)

### Sử dụng các phương thức của lớp `BaseMessage` 

Lớp BaseMessage cung cấp một số phương thức:

1. Tạo một phiên bản mới với nội dung đã được cập nhật:

In [13]:
new_message = message.create_new_instance("new test content")
print(isinstance(new_message, BaseMessage))

True


2. Chuyển đổi thành một đối tượng OpenAIMessage :


In [14]:
from camel.types import OpenAIBackendRole
openai_message = message.to_openai_message(role_at_backend=OpenAIBackendRole.USER)
print(openai_message == {"role": "user", "content": "test content"})

True


3. Chuyển đổi thành một đối tượng `OpenAISystemMessage` :

In [15]:
openai_system_message = message.to_openai_system_message()
print(openai_system_message == {"role": "system", "content": "test content"})

True


4. Converting to an `OpenAIUserMessage` object:

In [16]:
openai_user_message = message.to_openai_user_message()
print(openai_user_message == {"role": "user", "content": "test content"})

True


5. Converting to an `OpenAIAssistantMessage` object:

In [17]:
openai_assistant_message = message.to_openai_assistant_message()
print(openai_assistant_message == {"role": "assistant", "content": "test content"})

True


6. Converting to a dictionary:

In [21]:
message_dict = message.to_dict()
message_dict

{'role_name': 'test_user',
 'role_type': 'user',
 'content': 'test content',
 'image_detail': 'auto',
 'video_detail': 'auto'}

Các phương thức này cho phép bạn chuyển đổi một phiên bản `BaseMessage` thành các loại tin nhắn khác nhau tùy theo nhu cầu của bạn.

### Give `BaseMessage` to `ChatAgent`

In [22]:
from io import BytesIO

import requests
from PIL import Image

from camel.agents import ChatAgent
from camel.messages import BaseMessage
# URL of the image
url = "https://raw.githubusercontent.com/camel-ai/camel/master/misc/logo_light.png"
response = requests.get(url)
img = Image.open(BytesIO(response.content))

# Define system message
sys_msg = BaseMessage.make_assistant_message(
    role_name="Assistant",
    content="You are a helpful assistant.",
)

# Set agent
camel_agent = ChatAgent(system_message=sys_msg, output_language="Vietnamese")

# Set user message
user_msg = BaseMessage.make_user_message(
    role_name="User", content="""what's in the image?""", image_list=[img]
)

# Get response information
response = camel_agent.step(user_msg)
print(response.msgs[0].content)

Hình ảnh chứa một hình ảnh của lạc đà được minh họa, bên cạnh đó là chữ "CAMEL-AI" với kiểu chữ màu tím.


# 🌟 Highlights

Cuốn sổ tay này đã hướng dẫn bạn thiết lập và chuyển đổi `BaseMessage` thành các loại tin nhắn khác nhau. Các thành phần này đóng vai trò thiết yếu trong hệ thống trò chuyện CAMEL, hỗ trợ việc tạo, quản lý và diễn giải tin nhắn một cách rõ ràng.

Các công cụ chính được sử dụng trong notebook này bao gồm:
- `CAMEL`: Một khung làm việc đa tác nhân mạnh mẽ hỗ trợ Retrieval-Augmented Generation và các kịch bản đóng vai đa tác nhân, cho phép thực hiện các nhiệm vụ phức tạp do AI điều khiển.
- `BaseMessage`: Lớp cơ sở cho các đối tượng tin nhắn được sử dụng trong hệ thống trò chuyện CAMEL. Lớp này được thiết kế để cung cấp cấu trúc nhất quán cho các tin nhắn trong hệ thống và cho phép chuyển đổi dễ dàng giữa các loại tin nhắn khác nhau.
